# Agent Network Transaction Cleanup
Engagement 2 · Mobile money agent network. Audit, clean, merge and summarise the legacy transaction extract.

Regenerate the raw files with `python generate_data.py` before running.

In [1]:
import pandas as pd
from cleaning_pipeline import clean_agent_data
pd.set_option("display.width", 200, "display.max_colwidth", 80)

## 1. Load and audit the raw data
The raw files are read **twice**: once with pandas defaults, and once with `keep_default_na=False` so that the text `"N/A"` and truly empty cells can be counted separately (pandas converts both to `NaN` by default).

In [2]:
txns = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")
raw = pd.read_csv("agent_transactions.csv", keep_default_na=False)   # everything kept as text

print("transactions:", txns.shape, "| agents:", agents.shape)
txns.info()

transactions: (612, 5) | agents: (20, 3)
<class 'pandas.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   txn_id    612 non-null    str    
 1   agent_id  612 non-null    str    
 2   txn_type  612 non-null    str    
 3   amount    590 non-null    float64
 4   txn_date  612 non-null    str    
dtypes: float64(1), str(4)
memory usage: 24.0 KB


In [3]:
amt_num = pd.to_numeric(raw["amount"], errors="coerce")
valid_types = ["Deposit", "Withdrawal"]
q1, q3 = amt_num.quantile([.25, .75]); iqr = q3 - q1
hi = q3 + 1.5 * iqr

audit = pd.DataFrame([
    ("Duplicate transactions (repeated txn_id)", "Uniqueness", "txn_id", raw["txn_id"].duplicated().sum()),
    ("txn_type with wrong case / trailing whitespace", "Consistency", "txn_type", (~raw["txn_type"].isin(valid_types)).sum()),
    ('Text "N/A" in amount', "Validity", "amount", (raw["amount"] == "N/A").sum()),
    ("Empty amount", "Completeness", "amount", (raw["amount"] == "").sum()),
    ('txn_date uses "/" instead of ISO "-"', "Validity", "txn_date", raw["txn_date"].str.contains("/").sum()),
    ("Extreme amount of exactly 250,000", "Accuracy", "amount", (raw["amount"] == "250000").sum()),
    (f"Amount above IQR upper fence ({hi:,.0f})", "Accuracy", "amount", (amt_num > hi).sum()),
    ("agent_id not in agent register", "Integrity", "agent_id", (~raw["agent_id"].isin(agents["agent_id"])).sum()),
], columns=["issue", "dimension", "column", "rows_affected"])
audit

,issue,dimension,column,rows_affected
0,Duplicate transactions (repeated txn_id),Uniqueness,txn_id,12
1,txn_type with wrong case / trailing whitespace,Consistency,txn_type,384
2,"Text ""N/A"" in amount",Validity,amount,7
3,Empty amount,Completeness,amount,15
4,"txn_date uses ""/"" instead of ISO ""-""",Validity,txn_date,5
5,"Extreme amount of exactly 250,000",Accuracy,amount,5
6,"Amount above IQR upper fence (6,044)",Accuracy,amount,40
7,agent_id not in agent register,Integrity,agent_id,8


**Reading the audit.**
- The counts are on the raw 612 rows, so they include the repeated copies (e.g. 7 `N/A` text values = 5 original rows + 2 duplicated copies).
- Of the 250,000 values, all are above the fence, but the fence also catches many ordinary-looking right-tail amounts (see section 6).
- `Validity` and `Completeness` overlap for `amount`: text `N/A` and empty cells both end up as missing after conversion.

## 2. Run the cleaning pipeline
Every rule lives in `cleaning_pipeline.py`; nothing is edited by hand and no row numbers are hard-coded.

In [4]:
clean, log = clean_agent_data(txns, agents)
log

,step,issue,dimension,column,rows_affected,action
0,1,Duplicate transactions (repeated txn_id),Uniqueness,txn_id,12,"Dropped extra copies, kept one row per txn_id"
1,2,"Inconsistent txn_type spelling, case or whitespace",Consistency,txn_type,376,"Standardised to ""Deposit"" / ""Withdrawal"""
2,3,"Non-numeric amount text (e.g. ""N/A"")",Validity,amount,0,"to_numeric(errors='coerce'); 0 when the CSV reader already turned ""N/A"" into..."
3,3,Missing amount,Completeness,amount,20,Kept as missing until step 5
4,3,"Mixed date separator (""/"" instead of ""-"")",Validity,txn_date,5,"Normalised, parsed as datetime, stored ISO"
5,4,"Amount outside IQR fences (-1,928 to 6,084)",Accuracy,amount,40,"Flagged in is_outlier, values kept"
6,5,Missing/invalid amount imputed,Completeness,amount,20,Median of valid non-outlier amounts by txn_type; flagged in amount_imputed
7,6,Transactions referencing agents not in register,Integrity,agent_id,7,"Kept (left join); county set to ""Unknown"", flagged unknown_agent"


In [5]:
print("Rows before:", len(txns), "| rows after:", len(clean), "| removed:", len(txns) - len(clean))
clean.head()

Rows before: 612 | rows after: 600 | removed: 12


,txn_id,agent_id,txn_type,amount,txn_date,is_outlier,amount_imputed,county,onboarded,unknown_agent
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,Nakuru,2024-09-01,False
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,Mombasa,2025-08-01,False
2,TX00003,AG008,Deposit,9129.0,2026-04-29,True,False,Kisumu,2024-08-01,False
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,Nakuru,2024-09-01,False
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,Nakuru,2024-09-01,False


## 3. Uniqueness, consistency, validity

In [6]:
print("duplicate txn_id after cleaning:", clean["txn_id"].duplicated().sum())
print("txn_type values:", sorted(clean["txn_type"].unique()))
print("amount dtype:", clean["amount"].dtype)
print("txn_date range:", clean["txn_date"].min(), "to", clean["txn_date"].max())

duplicate txn_id after cleaning: 0
txn_type values: ['Deposit', 'Withdrawal']
amount dtype: float64
txn_date range: 2026-03-01 to 2026-04-29


- **Uniqueness:** 12 duplicate `txn_id` rows were removed (612 → 600). Eleven were exact copies; one copy differed only in `agent_id` (the unknown-agent label was applied after the copies were made), so the pipeline keeps the copy whose agent exists in the register.
- **Consistency:** five spellings of two values were standardised to exactly `Deposit` and `Withdrawal` (after `strip()` and `lower()`).
- **Validity:** `amount` is now numeric and `txn_date` is parsed as a date after normalising the five `/` dates, then stored in ISO `YYYY-MM-DD` form.

## 4. Completeness: missing and invalid amounts

In [7]:
imputed = clean[clean["amount_imputed"]]
print("imputed rows:", len(imputed))
print("median used per type:"); print(imputed.groupby("txn_type")["amount"].first())

imputed rows: 20
median used per type:
txn_type
Deposit       1717.0
Withdrawal    1646.0
Name: amount, dtype: float64


**Decision: impute and flag.** 20 amounts (3.3% of rows) are missing or invalid (empty cells plus `"N/A"` text). Each was filled with the **median of valid, non-outlier amounts of the same `txn_type`**, and `amount_imputed = True` marks it.

*Why:* dropping them would remove 20 genuine transactions from the counts and understate activity; leaving them as `NaN` would silently zero them in sums. The median is robust to the very large values, so the 250,000 entries do not inflate the fill value.
*Caveat:* imputed values are estimates, not recorded money. Value totals should be quoted with and without imputed rows, and the flag lets the finance team exclude them. If the source system can supply the true amounts, they should replace the imputed ones.

## 5. Outliers (IQR method)

In [8]:
q1, q3 = clean.loc[~clean["amount_imputed"], "amount"].quantile([.25, .75])
iqr = q3 - q1
print(f"Q1={q1:,.0f}  Q3={q3:,.0f}  IQR={iqr:,.0f}  fences: {q1-1.5*iqr:,.0f} to {q3+1.5*iqr:,.0f}")
print("flagged:", clean["is_outlier"].sum())
big = clean[clean["amount"] == 250000]
print("rows at exactly 250,000:", len(big), "| share of total value:", f"{big['amount'].sum() / clean['amount'].sum():.1%}")
big[["txn_id", "agent_id", "county", "txn_type", "txn_date", "amount"]]

Q1=1,077  Q3=3,080  IQR=2,003  fences: -1,928 to 6,084
flagged: 40
rows at exactly 250,000: 5 | share of total value: 46.5%


,txn_id,agent_id,county,txn_type,txn_date,amount
306,TX00307,AG012,Kisumu,Deposit,2026-03-02,250000.0
315,TX00316,AG014,Nairobi,Deposit,2026-04-01,250000.0
424,TX00425,AG005,Mombasa,Withdrawal,2026-03-02,250000.0
463,TX00464,AG009,Nakuru,Withdrawal,2026-04-11,250000.0
594,TX00595,AG016,Kiambu,Deposit,2026-03-11,250000.0


In [9]:
flagged = clean[clean["is_outlier"]]
print("flagged but NOT 250,000:", (flagged["amount"] != 250000).sum(),
      "| their range:", flagged.loc[flagged["amount"] != 250000, "amount"].agg(["min", "max"]).tolist())

flagged but NOT 250,000: 35 | their range: [6158.0, 28847.0]


**Findings.**
- The IQR rule flags 40 amounts. Five are the injected **KES 250,000** values; the other 35 lie between about KES 6,100 and 29,000 and are the natural right tail of a skewed distribution (amounts are roughly log-normal, so a plain IQR rule over-flags).
- The five 250,000 entries share one identical round value, belong to five different agents in five different counties, and together make up **about 47% of all transaction value** (KES 1.25M of 2.69M). The identical value suggests a limit, placeholder or keying error rather than five real, independent transactions.

**Recommendation.** Do not delete and do not silently correct them. Keep them in the file with `is_outlier = True`, and ask the five agents / the legacy-system owner to confirm each against the original receipt or float ledger. Until confirmed, report totals both with and without them. The 35 moderate flags are plausible and should stay in headline figures; a log-scale IQR would be a better detector going forward.

## 6. Merge validation

In [10]:
print("transactions before merge:", len(clean))
print("agent register rows:", len(agents))
print("rows after merge:", len(clean), "(left join, many-to-one)")
print("transactions with unknown agents:", clean["unknown_agent"].sum())
print("unknown agent ids:", clean.loc[clean["unknown_agent"], "agent_id"].unique())
print("county counts:"); print(clean["county"].value_counts())

transactions before merge: 600
agent register rows: 20
rows after merge: 600 (left join, many-to-one)
transactions with unknown agents: 7
unknown agent ids: ['AG099']
county counts:
county
Nairobi    180
Mombasa    133
Kisumu     111
Nakuru      99
Kiambu      70
Unknown      7
Name: count, dtype: int64


**Unknown agents.** 7 transactions (after de-duplication; 8 on the raw file) reference `AG099`, which is not in the 20-agent register. A **left join** keeps them so no valid transaction is lost; their county is set to `"Unknown"` and `unknown_agent = True`. They are shown as their own line in the summaries (about KES 27k, 1% of value) rather than being assigned to a county by guesswork. The register owner should confirm whether AG099 is a new, decommissioned or mistyped agent.

## 7. County summary

In [11]:
county_summary = (clean.groupby(["county", "txn_type"])["amount"]
                  .agg(total_value="sum", n_transactions="count", avg_value="mean")
                  .round(0).reset_index())
county_summary

,county,txn_type,total_value,n_transactions,avg_value
0,Kiambu,Deposit,334737.0,43,7785.0
1,Kiambu,Withdrawal,76920.0,27,2849.0
2,Kisumu,Deposit,436242.0,74,5895.0
3,Kisumu,Withdrawal,76244.0,37,2061.0
4,Mombasa,Deposit,148748.0,72,2066.0
5,Mombasa,Withdrawal,368592.0,61,6042.0
6,Nairobi,Deposit,543352.0,118,4605.0
7,Nairobi,Withdrawal,161514.0,62,2605.0
8,Nakuru,Deposit,184771.0,60,3080.0
9,Nakuru,Withdrawal,328464.0,39,8422.0


In [12]:
(clean.groupby("county")["amount"].agg(total_value="sum", n_transactions="count", avg_value="mean")
      .round(0).sort_values("total_value", ascending=False))

,total_value,n_transactions,avg_value
county,,,
Nairobi,704866.0,180,3916.0
Mombasa,517340.0,133,3890.0
Nakuru,513235.0,99,5184.0
Kisumu,512486.0,111,4617.0
Kiambu,411657.0,70,5881.0
Unknown,27156.0,7,3879.0


## 8. Weekly transaction value by county

In [13]:
d = pd.to_datetime(clean["txn_date"])
clean["week_start"] = (d - pd.to_timedelta(d.dt.weekday, unit="D")).dt.strftime("%Y-%m-%d")   # Monday
weekly = clean.pivot_table(index="county", columns="week_start", values="amount",
                           aggfunc="sum", fill_value=0, margins=True, margins_name="Total").round(0)
weekly

week_start,2026-02-23,2026-03-02,2026-03-09,2026-03-16,2026-03-23,2026-03-30,2026-04-06,2026-04-13,2026-04-20,2026-04-27,Total
county,,,,,,,,,,,
Kiambu,0.0,18317.0,267641.0,22054.0,11504.0,22609.0,15030.0,12265.0,24507.0,17730.0,411657.0
Kisumu,1893.0,287933.0,23186.0,27935.0,27077.0,41077.0,15341.0,29297.0,37439.0,21308.0,512486.0
Mombasa,6759.0,285610.0,30418.0,25085.0,17050.0,38344.0,47033.0,26730.0,16687.0,23624.0,517340.0
Nairobi,7046.0,68967.0,52801.0,43048.0,41314.0,282931.0,56727.0,56302.0,64926.0,30804.0,704866.0
Nakuru,2684.0,56574.0,28730.0,39795.0,18868.0,18145.0,276609.0,26093.0,30862.0,14875.0,513235.0
Unknown,0.0,0.0,0.0,1639.0,0.0,8507.0,9824.0,0.0,1877.0,5309.0,27156.0
Total,18382.0,717401.0,402776.0,159556.0,115813.0,411613.0,420564.0,150687.0,176298.0,113650.0,2686740.0


**Business reading.**
- **Nairobi leads** on both value (KES 705k, 26% of the total) and volume (180 transactions, 30%), but its lead is helped by one 250,000 entry. Without the five 250,000 values the order is Nairobi 455k, Mombasa 267k, Nakuru 263k, Kisumu 262k, Kiambu 162k: Nairobi still leads, but Mombasa, Nakuru and Kisumu are within 5k of each other, so their order is not reliable.
- **Deposit/withdrawal averages are distorted by outliers:** Mombasa and Nakuru withdrawals average 6,042 and 8,422 against about 2,000 to 3,000 for the other types, because each holds a 250,000 withdrawal; Kiambu's deposit average (7,785) is inflated the same way.
- **Weekly pivot:** each county has one week with a spike of about 250k (Kiambu w/c 9 Mar, Kisumu and Mombasa w/c 2 Mar, Nairobi w/c 30 Mar, Nakuru w/c 6 Apr). These are the flagged outliers, not real demand peaks; typical weekly county value is 15k to 65k. The first week (w/c 23 Feb) is partial, because data starts on Sunday 1 March.

## 9. Save outputs

In [14]:
clean_out = clean.drop(columns="week_start")
clean_out.to_csv("outputs/agent_transactions_clean.csv", index=False)
chk = pd.read_csv("outputs/agent_transactions_clean.csv")
print(chk.shape, "| duplicate txn_id:", chk["txn_id"].duplicated().sum(), "| types:", sorted(chk["txn_type"].unique()))

(600, 10) | duplicate txn_id: 0 | types: ['Deposit', 'Withdrawal']


# Data Quality Report

**Scope.** 612 raw transactions from 20 registered agents (plus one unregistered), 1 March – 29 April 2026. After cleaning: **600 transactions**, 12 removed, no valid data lost. Every step is reproducible via `clean_agent_data()`.

| Dimension | Issue found | Rows | Treatment |
|---|---|---|---|
| **Uniqueness** | Repeated `txn_id` | 12 | Removed; one copy kept per ID (prefer registered agent). |
| **Consistency** | `txn_type` in 5 spellings (case, trailing space) | 376 | Mapped to `Deposit` / `Withdrawal`. |
| **Validity** | `amount` held as text incl. `"N/A"`; 5 dates with `/` separator | 20 / 5 | `to_numeric`; separators normalised, parsed to datetime, saved as ISO. |
| **Completeness** | Missing or invalid `amount` | 20 (3.3%) | Imputed with the median valid non-outlier amount by type; flagged `amount_imputed`. |
| **Accuracy** | 40 amounts beyond IQR fences, incl. five KES 250,000 | 40 | Flagged `is_outlier`, retained; 250,000 values referred for verification (about 47% of total value). |
| **Integrity** (referential) | `AG099` not in the agent register | 7 | Left join keeps rows; county `Unknown`; flagged `unknown_agent`. |

**Impact on analysis.** County totals are sensitive to the 250,000 entries (they add KES 1.25M) and to imputed values (20 rows). Headline figures should be reported with and without them until verified.

**Remaining risks / next steps.** (1) Verify the five 250,000 transactions with source records. (2) Obtain true amounts for the 20 missing values. (3) Confirm the status of `AG099`. (4) Fix the legacy export to enforce a numeric `amount`, ISO dates, a controlled `txn_type` list and a primary-key constraint on `txn_id`.

### Privacy: personal data and the Kenya Data Protection Act 2019
This extract has no customer identifiers, but if it also contained **customer phone numbers**, then the phone number itself would be personal data (it identifies a person directly), and so would any field that can single someone out when combined with it: the customer's name or ID number, the agent-customer transaction history, amounts and timestamps that reveal location or behaviour (the `agent_id` and `county` fields indicate where a person transacted). Mobile-money records can also expose financial status, so they warrant extra care. Under the Data Protection Act 2019 I would: (1) collect and keep only what the analysis needs (**data minimisation**); (2) **pseudonymise** the phone number with a salted hash or a surrogate key before analysis and keep the lookup table separately under restricted access; (3) process it only for the stated purpose, on a lawful basis, with notice to data subjects; (4) apply **role-based access**, encryption in transit and at rest, and no sharing of raw extracts by email or in the Git repository; (5) set a **retention limit** and delete data afterwards; (6) report on aggregated data wherever possible; and (7) have the provider (the data controller) register with the Office of the Data Protection Commissioner, keep records of processing, and follow the breach-notification duty (72 hours).